# 06. MCP 서버 기초 (FastMCP 서버 만들기)

> **왜 MCP를 배워야 하나요?**
>
> 에이전트에게 도구를 제공하는 방식이 프로젝트마다 다르면 코드 재사용이 어려워요. MCP는 도구를 **표준화된 방식**으로 패키징해서 어떤 에이전트에서든 즉시 사용할 수 있게 해줘요. 한 번 만든 MCP 서버는 LangChain, Claude, Codex 등 어떤 프레임워크에서도 동일하게 동작해요.

> 🔑 **비유**: MCP는 **USB 포트**와 같아요. USB 이전에는 프린터, 마우스, 키보드마다 다른 포트를 사용했어요. USB가 나온 후에는 어떤 기기든 하나의 포트로 연결할 수 있게 되었죠. MCP가 AI 도구의 세계에서 같은 역할을 해요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. MCP(Model Context Protocol)의 개념과 아키텍처를 설명할 수 있어요
2. FastMCP로 `@mcp.tool()` 데코레이터를 사용해 도구를 정의할 수 있어요
3. **stdio** 전송 방식으로 로컬 MCP 서버를 만들고 실행할 수 있어요
4. **streamable-http** 전송 방식으로 원격 MCP 서버를 만들고 실행할 수 있어요
5. MCP Inspector를 활용해 서버를 테스트하고 디버깅할 수 있어요

## 사전 지식

- Part 05의 `02-Tools-V1.ipynb`에서 배운 `@tool` 데코레이터와 도구 바인딩
- Python 비동기 함수(`async def`, `await`) 기초
- LangChain 도구(Tool) 개념

## MCP란 무엇인가요?

**MCP(Model Context Protocol)**는 AI 애플리케이션이 언어 모델에 도구(Tool)와 컨텍스트를 제공하는 방법을 표준화한 오픈 프로토콜이에요.

기존 방식에서는 각 도구마다 개별적인 연동 코드를 작성해야 했어요. 날씨 API, 데이터베이스, 파일 시스템, 외부 서비스 등을 연결할 때마다 서로 다른 방식으로 구현해야 했죠. MCP는 이 모든 것을 **하나의 표준 인터페이스**로 통합해요.

> 🔑 **핵심 개념**: MCP는 USB 포트와 같아요. USB가 어떤 기기든 하나의 표준 방식으로 컴퓨터에 연결할 수 있게 해주듯, MCP는 어떤 도구든 표준 방식으로 LLM에 연결할 수 있게 해줘요.

### MCP의 주요 특징

| 특징 | 설명 |
|------|------|
| 표준화된 도구 인터페이스 | 어떤 서비스든 동일한 방식으로 도구를 정의하고 노출해요 |
| 다양한 전송 메커니즘 | stdio, HTTP(streamable-http) 등 여러 통신 방식을 지원해요 |
| 동적 도구 검색 | 클라이언트가 런타임에 서버의 도구 목록을 자동으로 발견해요 |
| 언어 독립적 | Python, Node.js, Go 등 어떤 언어로도 서버/클라이언트를 구현할 수 있어요 |

### MCP 생태계

```mermaid
flowchart LR
    A["MCP 서버 레지스트리<br>(Smithery AI)"] -->|"공개된 서버<br>목록"| B
    
    subgraph B["MCP 서버들"]
        S1["날씨 서버<br>(stdio)"]
        S2["시간 서버<br>(HTTP)"]
        S3["Context7 서버<br>(npx)"]
        S4["커스텀 서버<br>(FastMCP)"]
    end
    
    B --> C["MCP 클라이언트<br>(MultiServerMCPClient)"]
    C --> D["LLM 에이전트"]
    D --> E["사용자"]
    
    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    
    class A storage
    class S1,S2,S3,S4 process
    class C process
    class D input
    class E output
```

## MCP 전송 방식 비교

MCP 서버는 두 가지 주요 전송(transport) 방식을 지원해요.

```mermaid
flowchart TB
    subgraph STDIO["stdio 전송 방식"]
        direction LR
        C1["MCP 클라이언트"] -->|"stdin/stdout"| S1["서버 프로세스<br>(자동 관리)"]
    end
    
    subgraph HTTP["streamable-http 전송 방식"]
        direction LR
        C2["MCP 클라이언트"] -->|"HTTP POST /mcp"| S2["서버 프로세스<br>(독립 실행)"]
    end
    
    STDIO -->|"특징: 로컬, 클라이언트가 프로세스 관리"| NOTE1[" "]
    HTTP -->|"특징: 원격, 서버가 독립적으로 실행"| NOTE2[" "]
    
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef note fill:#fff3cd,stroke:#ffc107,color:#856404
    
    class C1,C2,S1,S2 process
    class NOTE1,NOTE2 note
```

| 전송 방식 | 사용 시점 | 서버 실행 | 엔드포인트 |
|-----------|----------|----------|----------|
| **stdio** | 로컬 개발, 테스트 | 클라이언트가 자동 시작 | 표준 입출력 |
| **streamable-http** | 원격 서버, 프로덕션 | 별도 프로세스로 독립 실행 | `http://host:port/mcp` |

> 💡 **핵심 정리**: 두 전송 방식의 차이를 실제로 시연해볼게요. stdio는 클라이언트가 서버 프로세스를 자식 프로세스로 자동 생성하기 때문에, 클라이언트만 실행하면 돼요. HTTP는 서버를 먼저 별도로 띄워야 해요.

## 환경 설정

In [ ]:
# 환경 변수 로드 (.env 파일에서 API 키를 읽어와요)
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# LangSmith 추적 설정 (선택 사항 - 에이전트 실행 과정을 추적해요)
import os

# LangSmith 추적을 활성화해요
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-V1-MCP-Basics"

## 1. FastMCP 소개

**FastMCP**는 Python으로 MCP 서버를 만들 수 있는 고수준 라이브러리예요. FastAPI와 유사한 데코레이터 기반 문법을 사용해서 도구를 아주 간결하게 정의할 수 있어요.

> 🔑 **핵심 개념**: FastMCP 서버의 구조는 딱 세 가지예요:
> 1. `FastMCP("서버이름")` — 서버 인스턴스 생성
> 2. `@mcp.tool()` — 도구 등록 (함수에 붙이는 데코레이터)
> 3. `mcp.run(transport="...")` — 서버 실행

### FastMCP 기본 구조

```python
from mcp.server.fastmcp import FastMCP

# 1. 서버 생성
mcp = FastMCP("서버이름", instructions="서버 설명")

# 2. 도구 정의
@mcp.tool()
async def my_tool(param: str) -> str:
    """도구 설명 (LLM이 이 docstring을 읽고 언제 이 도구를 쓸지 결정해요)"""
    return f"결과: {param}"

# 3. 서버 실행
if __name__ == "__main__":
    mcp.run(transport="stdio")  # 또는 "streamable-http"
```

> 💡 **실무 팁**: `@mcp.tool()` 함수의 **docstring**이 매우 중요해요. LLM은 docstring을 읽고 이 도구를 언제, 어떻게 사용할지 결정하기 때문에, 명확하고 구체적인 설명을 작성하면 도구 선택 정확도가 높아져요.

### FastMCP 설치 확인

In [ ]:
import mcp
from mcp.server.fastmcp import FastMCP
import importlib.metadata

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: mcp 패키지가 설치되어 있는지 확인해요
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 2. stdio 전송 방식 — 날씨 서버

첫 번째 MCP 서버를 만들어볼게요. **stdio 방식**은 클라이언트가 서버 Python 파일을 서브프로세스로 자동 실행하고, 표준 입출력(stdin/stdout)을 통해 통신해요.

아래 셀은 날씨 서버 파일을 `servers/01_weather_server.py`에 작성해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


서버 파일을 직접 읽어서 내용을 확인해볼게요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### stdio 방식으로 MCP 서버에 연결하기

이제 `MultiServerMCPClient`를 사용해서 방금 만든 날씨 서버에 연결해볼게요.

> 💡 **핵심 정리**: stdio 방식의 핵심은 `"command"`와 `"args"`예요. 클라이언트가 `uv run python servers/01_weather_server.py` 명령을 자동으로 실행해서 서버 프로세스를 관리해요. 터미널에서 서버를 별도로 켜지 않아도 돼요!

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from concurrent.futures import ThreadPoolExecutor
import asyncio

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 날씨 도구를 직접 호출해보기

에이전트를 거치지 않고 MCP 도구를 직접 호출해서 동작을 확인해볼게요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


> 💡 **실무 팁**: 도구를 직접 호출하는 방식은 MCP 서버 개발 중 빠른 테스트에 유용해요. 에이전트 전체를 실행하지 않아도 도구가 올바르게 동작하는지 확인할 수 있어요.

## 3. streamable-http 전송 방식 — 시간 서버

두 번째 전송 방식인 **streamable-http**를 사용하는 시간 서버를 만들어볼게요. 이 방식은 서버를 별도의 프로세스로 독립 실행한 뒤, 클라이언트가 HTTP로 연결해요.

> ⚠️ **자주 하는 실수**: HTTP 방식 서버는 반드시 **클라이언트 연결 전에** 서버를 먼저 실행해야 해요. 서버가 실행 중이지 않으면 연결 오류가 발생해요. 아래 셀에서 서버 파일을 만든 다음, **터미널에서 별도로 서버를 실행**해야 해요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### HTTP 방식 서버에 연결하기

**먼저 터미널에서 서버를 실행해야 해요:**
```bash
uv run python servers/02_time_server.py
```

서버가 실행된 후 아래 셀을 실행하세요.

> 🔑 **핵심 개념**: HTTP 방식은 `"command"`/`"args"` 대신 `"url"`을 사용해요. 서버 주소를 직접 지정하기 때문에, 원격 서버에도 연결할 수 있어요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 기본 시간대 (서울)
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


> ⚠️ **자주 하는 실수**: 클라이언트 설정에서 `"streamable_http"` (밑줄)을 사용해요. 서버 코드의 `mcp.run(transport="streamable-http")` (하이픈)과 헷갈리지 않도록 주의하세요.

## 4. 계산기 서버 — 직접 만들어보기

지금까지 날씨 서버(stdio)와 시간 서버(HTTP)를 만들었어요. 이번에는 여러 도구를 가진 계산기 서버를 만들어볼게요. 하나의 MCP 서버에 여러 도구를 등록하는 방법도 배울 수 있어요.

계산기 서버는 `add`, `subtract`, `multiply`, `divide` 네 개의 수학 연산 도구를 제공해요.

> 💡 **핵심 정리**: `@mcp.tool()` 데코레이터를 여러 함수에 붙이면 하나의 서버에서 여러 도구를 제공할 수 있어요. 클라이언트는 서버에 연결하면 자동으로 모든 도구 목록을 받아와요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


### 계산기 서버 도구 목록 확인하기

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 5. 여러 서버를 동시에 연결하기

`MultiServerMCPClient`의 가장 강력한 기능은 **여러 서버를 동시에 연결**해서 모든 도구를 하나의 목록으로 통합할 수 있다는 거예요.

> 🔑 **핵심 개념**: 에이전트 입장에서는 날씨 서버에서 온 도구인지, 계산기 서버에서 온 도구인지 구분할 필요가 없어요. `client.get_tools()`를 호출하면 연결된 모든 서버의 도구가 하나의 리스트로 합쳐져요.

In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 6. MCP Inspector — 브라우저에서 서버 테스트하기

**MCP Inspector**는 MCP 서버를 브라우저에서 시각적으로 테스트하고 디버깅할 수 있는 공식 도구예요.

### 사용 방법

터미널에서 다음 명령을 실행하면 Inspector가 시작돼요:

```bash
# npx로 실행해요 (Node.js가 필요해요)
npx @modelcontextprotocol/inspector
```

브라우저에서 `http://localhost:6274`로 접속하면 Inspector UI가 열려요.

### Inspector에서 서버 연결하기

1. **Transport**: `STDIO` 또는 `Streamable HTTP` 선택
2. **Command** (stdio): `python servers/01_weather_server.py` 입력
3. **URL** (HTTP): `http://127.0.0.1:8002/mcp` 입력
4. **Connect** 버튼 클릭
5. **Tools 탭**에서 도구 목록 확인 및 직접 호출 가능

> 💡 **실무 팁**: MCP Inspector는 서버 개발 초기에 도구가 올바르게 등록되었는지, docstring이 제대로 노출되는지 확인하기에 최적의 도구예요. 에이전트 없이 빠르게 기능을 검증할 수 있어요.

> 💡 **핵심 정리**: 라이브로 Inspector를 실행해서 날씨 서버의 `get_weather` 도구를 직접 호출해볼게요. `location` 파라미터에 여러 도시 이름을 입력해보면서 서버 응답을 확인해요.

## 7. 실습: 간단한 메모 서버 만들기

지금까지 배운 내용을 바탕으로 직접 MCP 서버를 만들어볼 차례예요. 메모를 저장하고 조회하는 간단한 서버를 만들어봐요.

아래 TODO 블록에 코드를 작성해보세요.

In [ ]:
from mcp.server.fastmcp import FastMCP

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


In [ ]:
# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: 메모 서버 직접 실행 및 테스트
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 8. 서버 구조 정리

이 노트북에서 만든 세 가지 MCP 서버의 구조를 정리해볼게요.

```mermaid
flowchart LR
    subgraph SERVERS["생성된 MCP 서버"]
        direction TB
        W["01_weather_server.py<br>---<br>transport: stdio<br>도구: get_weather"]
        T["02_time_server.py<br>---<br>transport: streamable-http<br>포트: 8002<br>도구: get_current_time"]
        C["03_calculator_server.py<br>---<br>transport: stdio<br>도구: add, subtract<br>       multiply, divide"]
    end
    
    subgraph CLIENT["클라이언트 설정"]
        direction TB
        CS1["command + args<br>→ subprocess 자동 관리"]
        CS2["url<br>→ HTTP 연결"]
    end
    
    W & C --> CS1
    T --> CS2
    CS1 & CS2 --> M["MultiServerMCPClient<br>.get_tools()"]
    M --> AG["LLM 에이전트<br>(다음 노트북에서)"]
    
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    
    class W,T,C process
    class CS1,CS2 storage
    class M,AG output
```

### 만들어진 서버 파일 목록 확인

In [ ]:
import os

# TODO: 이 셀의 핵심 구현 코드를 수업 시간에 직접 작성하세요.
# 목표: ---------------------------------------------------
# 위 마크다운 설명과 힌트를 참고해 필요한 타입, 함수, 그래프 구성, 실행 코드를 완성하세요.


## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **MCP(Model Context Protocol)**: AI 애플리케이션에서 도구와 컨텍스트를 표준화하는 오픈 프로토콜이에요. USB처럼 어떤 도구든 하나의 표준 방식으로 LLM에 연결해요
- **FastMCP**: Python으로 MCP 서버를 빠르게 만들 수 있는 라이브러리예요. `FastMCP()` + `@mcp.tool()` + `mcp.run()` 세 단계로 서버를 만들어요
- **stdio 전송 방식**: 클라이언트가 서버를 서브프로세스로 자동 관리해요. `command`/`args`로 설정하고 로컬 개발에 적합해요
- **streamable-http 전송 방식**: 서버를 독립 프로세스로 실행하고 클라이언트가 HTTP로 연결해요. `url`로 설정하고 원격 서버나 프로덕션 환경에 적합해요
- **MultiServerMCPClient**: 여러 MCP 서버를 동시에 연결하고 모든 도구를 하나의 목록으로 통합해요
- **MCP Inspector**: `npx @modelcontextprotocol/inspector`로 실행하는 브라우저 기반 디버깅 도구예요

### 전송 방식 비교 요약

| 항목 | stdio | streamable-http |
|------|-------|----------------|
| 서버 실행 | 클라이언트가 자동 | 별도 프로세스로 수동 |
| 설정 키 | `command`, `args` | `url` |
| 적합한 환경 | 로컬 개발, 테스트 | 원격 서버, 프로덕션 |
| 클라이언트 설정 | `"transport": "stdio"` | `"transport": "streamable_http"` |

## 다음 노트북 예고

다음 `07-MCP-Agent-Integration.ipynb`에서는 **MultiServerMCPClient와 `create_agent` 연동**을 배워요. 이번 노트북에서 만든 MCP 서버들을 실제 LLM 에이전트에 연결해서, 에이전트가 MCP 도구를 자율적으로 선택하고 사용하는 방법을 학습해요.